## 1. Import Libraries & Load Data

In [344]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, classification_report, confusion_matrix, accuracy_score
# Load Data
data_path = "../../data/processed/formatted_data.csv"
try:
    df = pd.read_csv(data_path)
    print(f"Data loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    df = pd.read_csv("D:/AIO/WarmUP/AIO Conquer Warmup/aio-aiconquer-churn-prediction/data/processed/formatted_data.csv")


Data loaded successfully. Shape: (100076, 31)


## 2. Preprocessing

In [345]:
# Drop customer_id
if 'customer_id' in df.columns:
    df = df.drop(columns=['customer_id'])
    print("Dropped 'customer_id'.")

# --- BƯỚC MỚI: FEATURE ENGINEERING ---
# 1. Chi phí trung bình thực tế mỗi tháng (để tìm ra khách hàng bị tính phí ẩn/phạt)
# Cộng 1 để tránh lỗi chia cho 0
df['actual_monthly_cost'] = df['totalcharges'] / (df['tenure'] + 1)

# 2. Mức độ "đắt đỏ" của gói dữ liệu (Tỷ lệ cước phí so với dung lượng dùng)
df['cost_per_gb'] = df['monthlycharges'] / (df['avg_monthly_gb'] + 1)

# 3. Gom nhóm thời gian gắn bó (Tenure Binning) để mô hình dễ nhận dạng khách hàng mới/cũ
bins = [-1, 6, 24, 60, 1000]
labels = ['New', 'Steady', 'Loyal', 'Very_Loyal']
df['tenure_group'] = pd.cut(df['tenure'], bins=bins, labels=labels)

# Categorical columns specified for One-Hot Encoding
cat_cols = ['gender', 'education', 'marital_status', 'contract', 'payment_method', 'tenure_group']
# -------------------------------------

# One-Hot Encoding
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Convert boolean to int for XGBoost
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)
        
# Check NaNs
total_nans = df.isnull().sum().sum()
if total_nans > 0:
    print(f"Warning: {total_nans} NaN values found. Filling with median.")
    df = df.fillna(df.median())
else:
    print("Confirmed: No NaN values.")
    
# Split X and y
X = df.drop(columns=['churn'])
y = df['churn']

Dropped 'customer_id'.
Confirmed: No NaN values.


## 3. Train/Test Split (Stratified)

In [346]:
# 80/20 split, stratify=y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Test shapes: X={X_test.shape}, y={y_test.shape}")
print(f"Churn ratio in Train: {y_train.mean():.4f} | Test: {y_test.mean():.4f}")

Train shapes: X=(80060, 43), y=(80060,)
Test shapes: X=(20016, 43), y=(20016,)
Churn ratio in Train: 0.0999 | Test: 0.0999


## 4. Model Configuration & Training

In [347]:
# Using scale_pos_weight for imbalanced data
scale_pos_weight = 4.0

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 2,
    'min_child_weight': 4,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42
}
print(f"Hyperparameters: {params}")

model = xgb.XGBClassifier(**params, n_estimators=10000, early_stopping_rounds=50)

# Train
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print(f"\nBest iteration: {model.best_iteration}")

Hyperparameters: {'objective': 'binary:logistic', 'eval_metric': 'auc', 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 4, 'subsample': 0.8, 'colsample_bytree': 0.8, 'scale_pos_weight': 4.0, 'random_state': 42}
[0]	validation_0-auc:0.62509
[50]	validation_0-auc:0.65381
[100]	validation_0-auc:0.65578
[150]	validation_0-auc:0.66011
[200]	validation_0-auc:0.66408
[250]	validation_0-auc:0.66674
[300]	validation_0-auc:0.66915
[350]	validation_0-auc:0.67082
[400]	validation_0-auc:0.67210
[450]	validation_0-auc:0.67313
[500]	validation_0-auc:0.67381
[550]	validation_0-auc:0.67423
[600]	validation_0-auc:0.67471
[650]	validation_0-auc:0.67490
[700]	validation_0-auc:0.67527
[750]	validation_0-auc:0.67557
[800]	validation_0-auc:0.67579
[850]	validation_0-auc:0.67594
[900]	validation_0-auc:0.67606
[950]	validation_0-auc:0.67620
[1000]	validation_0-auc:0.67627
[1050]	validation_0-auc:0.67630
[1100]	validation_0-auc:0.67628
[1117]	validation_0-auc:0.67629

Best iteration: 1068


## 5. Evaluation & Threshold Tuning

In [348]:
y_pred_prob = model.predict_proba(X_test)[:, 1]

# Default threshold 0.5
print("--- Evaluation at default threshold (0.5) ---")
y_pred_05 = (y_pred_prob >= 0.5).astype(int)
print(classification_report(y_test, y_pred_05))

# Threshold Tuning
thresholds = np.arange(0.1, 0.9, 0.1)
best_thresh = 0.5
best_f1 = 0

print("\n--- Threshold Tuning Grid ---")
for t in thresholds:
    y_pred_t = (y_pred_prob >= t).astype(int)
    rec = recall_score(y_test, y_pred_t)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t)
    print(f"Threshold: {t:.1f} | Recall: {rec:.4f} | Precision: {prec:.4f} | F1: {f1:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        
print(f"\n=> Selected Best Threshold for F1-Score: {best_thresh:.1f}")

--- Evaluation at default threshold (0.5) ---
              precision    recall  f1-score   support

           0       0.91      0.96      0.93     18017
           1       0.27      0.12      0.17      1999

    accuracy                           0.88     20016
   macro avg       0.59      0.54      0.55     20016
weighted avg       0.84      0.88      0.86     20016


--- Threshold Tuning Grid ---
Threshold: 0.1 | Recall: 0.9980 | Precision: 0.1012 | F1: 0.1837
Threshold: 0.2 | Recall: 0.8869 | Precision: 0.1207 | F1: 0.2125
Threshold: 0.3 | Recall: 0.6823 | Precision: 0.1507 | F1: 0.2469
Threshold: 0.4 | Recall: 0.3672 | Precision: 0.1995 | F1: 0.2585
Threshold: 0.5 | Recall: 0.1246 | Precision: 0.2695 | F1: 0.1704
Threshold: 0.6 | Recall: 0.0305 | Precision: 0.4122 | F1: 0.0568
Threshold: 0.7 | Recall: 0.0030 | Precision: 0.6000 | F1: 0.0060
Threshold: 0.8 | Recall: 0.0005 | Precision: 1.0000 | F1: 0.0010

=> Selected Best Threshold for F1-Score: 0.4


## 6. Final Evaluation

In [349]:
print(f"--- Final Evaluation metrics (Threshold {best_thresh:.1f}) ---")
y_pred_final = (y_pred_prob >= best_thresh).astype(int)

print(f"Recall:    {recall_score(y_test, y_pred_final):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_final):.4f}")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_prob):.4f}")

print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred_final)}")

--- Final Evaluation metrics (Threshold 0.4) ---
Recall:    0.3672
Precision: 0.1995
F1-Score:  0.2585
Accuracy:  0.7896
ROC-AUC:   0.6764

Confusion Matrix:
[[15071  2946]
 [ 1265   734]]
